In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow import keras
from tensorflow.keras import layers

(X_train, y_train), (X_test, y_test) = (
    keras.datasets.fashion_mnist.load_data()
)

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", X_test.shape)
print("Testing labels:", y_test.shape)

In [ ]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    # First convolution block
    layers.Conv2D(
        32,
        kernel_size=(3, 3),
        activation="relu"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Second convolution block
    layers.Conv2D(
        64,
        kernel_size=(3, 3),
        activation="relu"
    ),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Classification block
    layers.Flatten(),

    layers.Dropout(0.3),

    layers.Dense(128, activation="relu"),

    layers.Dense(10, activation="softmax")
])

cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = cnn_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fashion-MNIST CNN Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fashion-MNIST CNN Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_loss, test_accuracy = cnn_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

predictions = cnn_model.predict(X_test)

y_pred = np.argmax(predictions, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.title("Fashion-MNIST CNN Confusion Matrix")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names
    )
)

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")

print("Path to dataset files:", path)

In [ ]:
pip install kagglehub

In [ ]:
from pathlib import Path

DATASET_PATH = Path(path)

csv_files = list(DATASET_PATH.rglob("styles.csv"))

if not csv_files:
    raise FileNotFoundError("styles.csv not found in downloaded dataset")

STYLES_PATH = csv_files[0]
IMAGE_DIR = STYLES_PATH.parent / "images"

print("Dataset directory:", DATASET_PATH)
print("Metadata file:", STYLES_PATH)
print("Image directory:", IMAGE_DIR)

print("Images directory exists:", IMAGE_DIR.exists())

In [ ]:
import pandas as pd

styles_df = pd.read_csv(
    STYLES_PATH,
    on_bad_lines="skip"
)

print("Dataset shape:", styles_df.shape)

styles_df.head()

In [ ]:
print("Total image files:", len(list(IMAGE_DIR.glob("*.jpg"))))

print("Sample image files:")

for image_file in list(IMAGE_DIR.glob("*.jpg"))[:5]:
    print(image_file.name)

In [ ]:
styles_df["image_path"] = styles_df["id"].apply(
    lambda x: IMAGE_DIR / f"{int(x)}.jpg"
)

styles_df = styles_df.dropna(
    subset=["articleType"]
).copy()

styles_df = styles_df[
    styles_df["image_path"].apply(lambda x: x.exists())
].copy()

print("Available product images:", len(styles_df))

In [ ]:
category_counts = styles_df["articleType"].value_counts()

print("Total categories:", len(category_counts))

category_counts.head(20)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

category_counts.head(20).plot(kind="bar")

plt.title("Top 20 Fashion Product Categories")
plt.xlabel("Product Category")
plt.ylabel("Number of Images")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
MIN_SAMPLES = 100

valid_categories = category_counts[
    category_counts >= MIN_SAMPLES
].index

product_df = styles_df[
    styles_df["articleType"].isin(valid_categories)
].copy()

print("Total product images:", len(product_df))
print("Selected categories:", product_df["articleType"].nunique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

product_df["label"] = label_encoder.fit_transform(
    product_df["masterCategory"]
)

class_names = label_encoder.classes_

print("Number of classes:", len(class_names))
print("\nCategory mapping:")

for index, category in enumerate(class_names):
    print(index, ":", category)

In [ ]:

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    product_df,
    test_size=0.30,
    random_state=42,
    stratify=product_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Testing records:", len(test_df))

In [ ]:
print("Training distribution:")
print(train_df["label"].value_counts().sort_index())

print("\nValidation distribution:")
print(val_df["label"].value_counts().sort_index())

print("\nTesting distribution:")
print(test_df["label"].value_counts().sort_index())

In [ ]:
from PIL import Image

plt.figure(figsize=(12, 8))

sample_df = train_df.sample(
    n=12,
    random_state=42
).reset_index(drop=True)

for i, row in sample_df.iterrows():
    image = Image.open(row["image_path"]).convert("RGB")

    plt.subplot(3, 4, i + 1)
    plt.imshow(image)
    plt.title(row["articleType"])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

NUM_CLASSES = len(class_names)

print("Image size:", IMG_HEIGHT, IMG_WIDTH)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)

In [ ]:
def load_and_preprocess_image(image_path, label):

    image = tf.io.read_file(image_path)

    image = tf.io.decode_jpeg(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

In [ ]:
def create_image_dataset(dataframe, training=False):

    image_paths = dataframe["image_path"].astype(str).values
    labels = dataframe["label"].values.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(dataframe), 5000),
            seed=42
        )

    dataset = dataset.map(
        load_and_preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            lambda image, label: (
                tf.image.random_flip_left_right(image),
                label
            ),
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset

In [ ]:
train_ds = create_image_dataset(
    train_df,
    training=True
)

val_ds = create_image_dataset(
    val_df,
    training=False
)

test_ds = create_image_dataset(
    test_df,
    training=False
)

In [ ]:
for images, labels in train_ds.take(1):

    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)

    print("Minimum pixel value:", tf.reduce_min(images).numpy())
    print("Maximum pixel value:", tf.reduce_max(images).numpy())

In [ ]:
plt.figure(figsize=(12, 8))

for images, labels in train_ds.take(1):

    for i in range(min(12, len(images))):

        plt.subplot(3, 4, i + 1)

        plt.imshow(images[i].numpy())

        plt.title(class_names[labels[i].numpy()])

        plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cnn_model = keras.Sequential([

    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    # Convolution block 1
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Convolution block 3
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    # Classification block
    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(NUM_CLASSES, activation="softmax")
])

cnn_model.summary()

In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("CNN compiled successfully.")

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()

if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

print("Project directory:", BASE_DIR)

In [ ]:
MODEL_DIR = BASE_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Model directory:", MODEL_DIR)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(
            MODEL_DIR / "best_product_cnn.keras"
        ),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# Train the model on the Fashion Product Images dataset
EPOCHS = 15

history = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
import numpy as np

# Actual labels
y_true = np.concatenate([
    labels.numpy()
    for _, labels in test_ds
])

# Predicted probabilities
y_prob = cnn_model.predict(test_ds)

# Convert probabilities into predicted class indices
y_pred = np.argmax(y_prob, axis=1)

print("Actual labels shape:", y_true.shape)
print("Predicted labels shape:", y_pred.shape)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision (weighted):", precision_score(y_true, y_pred, average="weighted", zero_division=0))
print("Recall (weighted):", recall_score(y_true, y_pred, average="weighted", zero_division=0))
print("F1 Score (weighted):", f1_score(y_true, y_pred, average="weighted", zero_division=0))

print("\nClassification Report:\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(16, 12))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.title("Fashion Product CNN - Confusion Matrix")

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
def predict_product_image(image_path):
    image = tf.io.read_file(str(image_path))

    image = tf.io.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    image = tf.cast(image, tf.float32) / 255.0

    # Add batch dimension
    image = tf.expand_dims(image, axis=0)

    # Predict probabilities
    probabilities = cnn_model.predict(image, verbose=0)[0]

    predicted_index = int(np.argmax(probabilities))

    predicted_category = class_names[predicted_index]

    confidence = float(probabilities[predicted_index]) * 100

    return predicted_category, confidence, probabilities

In [ ]:
sample_image_path = test_df.iloc[0]["image_path"]

predicted_category, confidence, probabilities = predict_product_image(
    sample_image_path
)

print("Predicted category:", predicted_category)
print(f"Confidence: {confidence:.2f}%")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

image = Image.open(sample_image_path)

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")

plt.title(
    f"Prediction: {predicted_category}\n"
    f"Confidence: {confidence:.2f}%"
)

plt.show()

In [ ]:
top_3_indices = np.argsort(probabilities)[-3:][::-1]

for index in top_3_indices:
    print(
        f"{class_names[index]}: "
        f"{probabilities[index] * 100:.2f}%"
    )

In [ ]:
import json

class_mapping = {
    int(index): str(category)
    for index, category in enumerate(class_names)
}

mapping_path = MODEL_DIR / "product_class_mapping.json"

with open(mapping_path, "w") as file:
    json.dump(class_mapping, file, indent=4)

print("Class mapping saved:", mapping_path)

In [ ]:
FINAL_MODEL_PATH = MODEL_DIR / "fashion_product_cnn.keras"

cnn_model.save(FINAL_MODEL_PATH)

print("Model saved at:", FINAL_MODEL_PATH)

In [ ]:
image_config = {
    "image_height": IMG_HEIGHT,
    "image_width": IMG_WIDTH,
    "channels": 3,
    "normalization": "pixel_values_divided_by_255",
    "num_classes": int(NUM_CLASSES)
}

config_path = MODEL_DIR / "product_image_config.json"

with open(config_path, "w") as file:
    json.dump(image_config, file, indent=4)

print("Image configuration saved:", config_path)

In [ ]:

loaded_model = keras.models.load_model(
    MODEL_DIR / "fashion_product_cnn.keras"
)

print("Saved model loaded successfully!")

print("Input shape:", loaded_model.input_shape)
print("Output shape:", loaded_model.output_shape)